# BTC 48-hour magnitude: IC-guided nested Random Forest
This notebook reviews and reproduces the saved experiment. It does not start training by default.

**Outer test:** November 2024, December 2024, and January 1–22, 2025 on the supplied prepared dataset. **Inner selection:** three preceding monthly walk-forward folds before each outer month. Each model trains on its preceding three calendar months; training labels and inner selection labels must mature before the relevant issuance.

All 522 masked F4Z inputs remain candidates. The magnitude tables of the Top 20 PDF supply 38 candidate names, 32 of which satisfy the outer-window admission rules. Fixed causal lags, rolling summaries, cross-window slopes/balances, family summaries, matured magnitude histories, training-only IC screening and feature interactions expand the search. Direction-table rankings are not used.

Training MAE/IC are in-sample diagnostics. Overall training includes overlapping fit windows and therefore repeats some timestamps. Overall test includes each outer timestamp exactly once. Historically exposed data and feature priors remain exposed; this is not an untouched confirmation set.

In [1]:
from pathlib import Path
import json
import pandas as pd
# Set this to the copied workspace root when running elsewhere.
WORKSPACE = Path.cwd()
if not (WORKSPACE / 'Code/nested_ic_rf.py').exists() and (WORKSPACE.parent / 'Code/nested_ic_rf.py').exists():
    WORKSPACE = WORKSPACE.parent
OUTPUT = WORKSPACE / 'Experiments/nested_ic_rf_v3'
assert (WORKSPACE / 'Code/nested_ic_rf.py').exists(), 'Set WORKSPACE to the folder containing Code, Report, and Dataset.'


## Overall training and testing scores
MAE and RMSE use original log basis points; lower is better. IC is time-series correlation against magnitude. Rank IC is Spearman correlation, Pearson IC is also shown. High IC does not automatically imply lower MAE.


In [2]:
overall = pd.read_csv(OUTPUT / 'overall_training_testing.csv')
display(overall)
display(pd.read_csv(OUTPUT / 'scoreboard.csv'))


,split,model,mae,rmse,rank_ic,pearson_ic,rows
0,train_in_sample_pooled,nested_raw_rf,103.613846,155.859853,0.784022,0.801994,12912
1,test_outer_unique,nested_raw_rf,183.677216,254.862330,0.037958,0.043325,3937
2,train_in_sample_pooled,nested_standalone_rf,104.002790,154.504328,0.836632,0.840612,12912
3,test_outer_unique,nested_standalone_rf,185.969227,249.633164,0.030073,0.047136,3937
4,train_in_sample_pooled,nested_selected,104.002790,154.504328,0.836632,0.840612,12912
5,test_outer_unique,nested_selected,185.969227,249.633164,0.030073,0.047136,3937


,model,mae,rmse,rank_ic,pearson_ic,rows
0,rolling_median,180.049258,251.795311,0.027214,0.102284,3937
1,previous_rf,180.234192,254.096136,-0.025905,-0.048187,3937
2,previous_hybrid,180.766433,257.158508,-0.062545,-0.139150,3937
3,frozen_pair,181.339211,258.620107,-0.047337,-0.147803,3937
4,nested_raw_rf,183.677216,254.862330,0.037958,0.043325,3937
5,nested_standalone_rf,185.969227,249.633164,0.030073,0.047136,3937
6,nested_selected,185.969227,249.633164,0.030073,0.047136,3937
7,fixed_M0,188.699504,275.610784,0.000000,NaN,3937


## Monthly training, inner-validation, and outer-test results

In [3]:
display(pd.read_csv(OUTPUT / 'train_validation_test_by_month.csv'))
display(pd.read_csv(OUTPUT / 'monthly_mae.csv'))
display(pd.read_csv(OUTPUT / 'monthly_rank_ic.csv'))


,month,split,model,mae,rmse,rank_ic,pearson_ic,rows
0,2024-11,test,nested_raw_rf,235.929485,321.921039,-0.236250,-0.239569,1440
1,2024-11,test,nested_standalone_rf,242.436686,317.133723,-0.179341,-0.076555,1440
2,2024-11,test,nested_selected,242.436686,317.133723,-0.179341,-0.076555,1440
3,2024-11,train_in_sample,nested_raw_rf,150.197683,214.793167,0.656504,0.652463,4320
4,2024-11,train_in_sample,nested_standalone_rf,113.069067,169.315422,0.829657,0.844004,4320
5,2024-11,train_in_sample,nested_selected,113.069067,169.315422,0.829657,0.844004,4320
6,2024-11,inner_validation,nested_raw_rf,178.014983,250.463908,-0.118355,-0.145793,4320
7,2024-11,inner_validation,nested_standalone_rf,173.323555,254.196164,0.104079,0.024134,4320
8,2024-11,inner_validation,nested_selected,173.323555,254.196164,0.104079,0.024134,4320
9,2024-12,test,nested_raw_rf,145.114078,198.448638,0.134595,0.140086,1488


,month,nested_raw_rf,nested_standalone_rf,nested_selected,frozen_pair,previous_rf,previous_hybrid,fixed_M0,rolling_median
0,2024-11,235.929485,242.436686,242.436686,225.636876,225.726091,225.315803,230.256956,224.476582
1,2024-12,145.114078,146.198555,146.198555,149.605245,148.635379,149.137452,158.420854,147.680987
2,2025-01,165.975214,164.032278,164.032278,164.918502,161.909811,163.831677,174.043312,164.378930


,Unnamed: 0,nested_raw_rf,nested_standalone_rf,nested_selected,frozen_pair,previous_rf,previous_hybrid,fixed_M0,rolling_median
0,2024-11,-0.236250,-0.179341,-0.179341,-0.162127,-0.070978,-0.157282,0.0,0.0
1,2024-12,0.134595,0.134595,0.134595,-0.151712,-0.056224,-0.149479,0.0,0.0
2,2025-01,0.285598,0.097636,0.097636,0.191938,0.091234,0.225581,0.0,0.0


## Selected recipe and IC evidence for each outer month

In [4]:
for path in sorted((OUTPUT / 'outer').glob('*_selection.json')):
    selection = json.loads(path.read_text())
    print(selection['outer'], 'standalone:', selection['selected_standalone'],
          'selected:', selection['selected_forecast'], 'hybrid:', selection['selected_is_hybrid'])
    display(pd.DataFrame(selection['scores']).T.sort_values('mae'))
    name = pd.Timestamp(selection['outer']).strftime('%Y-%m') + '__magnitude.csv'
    if (OUTPUT / 'ic' / name).exists():
        display(pd.read_csv(OUTPUT / 'ic' / name).head(20))


2024-11-01 00:00:00+00:00 standalone: rf_top3_ensemble selected: rf_top3_ensemble hybrid: False


,mae,rmse,rank_ic,pearson_ic,rows
raw_all,178.014983,250.463908,-0.118355,-0.145793,4320.0
ic64_robust_interactions,179.391526,253.950172,0.090531,0.011723,4320.0
engineered_ic64_sqrt,180.079749,260.951598,0.134703,0.071330,4320.0
engineered_ic64_log,180.497827,273.196136,0.152574,0.085833,4320.0
ic64_rank_interactions,182.579200,256.150506,0.101772,0.006493,4320.0
engineered_ic24,182.963708,261.135631,0.083582,0.027156,4320.0
raw_all_deeper,183.950248,245.715780,0.039220,0.046379,4320.0
report_only,185.563434,253.987845,-0.054402,-0.061645,4320.0
residual_ic64,186.073754,268.030954,-0.072806,-0.060233,4320.0
engineered_ic128,188.085231,259.201069,-0.038943,-0.057909,4320.0


,feature,rank_ic,block_mean_ic,block_std_ic,sign_agreement,screen_score,observations
0,MEAN168H__F4Z__funding__funding_persistence_pe...,-0.371593,-0.411943,0.066729,1.0,0.498168,4320
1,STD168H__F4Z__flow__flow_activity_rz__1h__value,-0.368087,-0.359288,0.075414,1.0,0.443769,4320
2,F4Z__liquidation__liq_activity_severity_histor...,-0.177833,-0.368047,0.111205,1.0,0.401385,4320
3,STD168H__F4Z__flow__flow_activity_rz__4h__value,-0.325831,-0.317658,0.071952,1.0,0.391921,4320
4,STD168H__F4Z__flow__flow_activity_percentile__...,-0.364418,-0.316661,0.171967,1.0,0.390569,4320
5,DELTA24H__F4Z__funding__funding_persistence_pe...,0.289391,0.284468,0.040385,1.0,0.352777,4050
6,SLOPE__F4Z__basis__basis_level_rz__5m__value__...,0.261134,0.256890,0.136582,1.0,0.308516,4320
7,HIST__q75_30d,-0.019078,-0.310623,0.070460,1.0,0.308347,4320
8,BALANCE__F4Z__basis__basis_level_rz__5m__value...,0.259773,0.248552,0.140664,1.0,0.299429,4320
9,HIST__median_30d,-0.046925,-0.284901,0.152099,1.0,0.281423,4320


2024-12-01 00:00:00+00:00 standalone: engineered_ic64_log selected: engineered_ic64_log hybrid: False


,mae,rmse,rank_ic,pearson_ic,rows
engineered_ic64_log,168.188200,235.463242,0.265332,0.295657,4272.0
engineered_ic24,172.933737,227.124159,0.234482,0.255070,4272.0
raw_all,174.566407,235.522427,-0.059246,-0.105214,4272.0
residual_ic64,176.919712,230.347775,0.162910,0.185205,4272.0
engineered_ic64,177.029887,229.767553,0.229977,0.243480,4272.0
engineered_ic128,177.928658,228.542534,0.178809,0.227356,4272.0
ic64_rank_interactions,178.134622,229.961371,0.219730,0.215459,4272.0
ic64_robust_interactions,179.168805,232.733106,0.166000,0.178026,4272.0
report_only,179.208138,243.096393,-0.037161,-0.045459,4272.0
engineered_ic64_sqrt,180.628166,238.463573,0.217253,0.197373,4272.0


,feature,rank_ic,block_mean_ic,block_std_ic,sign_agreement,screen_score,observations
0,F4Z__liquidation__liq_activity_severity_histor...,-0.136611,-0.340714,0.143699,1.000000,0.360497,4272
1,MEAN168H__F4Z__liquidation__liq_temporal_conce...,0.108849,0.318810,0.169707,1.000000,0.329052,4272
2,STD168H__F4Z__flow__flow_activity_rz__1h__value,-0.318707,-0.259481,0.189190,1.000000,0.320238,4272
3,STD168H__F4Z__volatility__realized_variance_le...,-0.275819,-0.260970,0.159186,1.000000,0.314006,4272
4,TIME__weekday_cos,0.256094,0.256165,0.149376,1.000000,0.305251,4272
5,STD168H__F4Z__volatility__realized_variance_le...,-0.282747,-0.244292,0.101472,1.000000,0.304832,4272
6,STD168H__F4Z__volatility__realized_variance_le...,-0.256152,-0.243507,0.104059,1.000000,0.297140,4272
7,STD168H__F4Z__volatility__realized_variance_le...,-0.231682,-0.243869,0.158325,1.000000,0.285957,4272
8,HIST__median_7d,-0.173265,-0.243620,0.075839,1.000000,0.279352,4272
9,STD168H__F4Z__flow__flow_activity_rz__4h__value,-0.218542,-0.234742,0.146093,1.000000,0.274768,4272


2025-01-01 00:00:00+00:00 standalone: rf_top3_ensemble selected: rf_top3_ensemble hybrid: False


,mae,rmse,rank_ic,pearson_ic,rows
engineered_ic64_log,171.892592,238.961302,0.167406,0.251139,4320.0
raw_all,174.015497,239.254793,-0.037243,-0.074166,4320.0
report_only,174.160921,243.780373,-0.032713,-0.039744,4320.0
residual_ic64,177.438080,234.919409,0.069907,0.122804,4320.0
ic64_robust_interactions,180.699666,237.766109,0.129164,0.142750,4320.0
ic64_rank_interactions,181.939650,237.336575,0.133545,0.149624,4320.0
engineered_ic64_sqrt,183.378035,242.852200,0.106699,0.129087,4320.0
engineered_ic64,184.657765,237.817798,0.119158,0.171571,4320.0
engineered_ic24,185.825019,239.381203,0.121180,0.159581,4320.0
engineered_ic128,186.520210,238.845649,0.144429,0.193809,4320.0


,feature,rank_ic,block_mean_ic,block_std_ic,sign_agreement,screen_score,observations
0,TIME__weekday_cos,0.327119,0.321580,0.088075,1.000000,0.394552,4320
1,STD168H__F4Z__volatility__realized_variance_le...,-0.324285,-0.305424,0.113978,1.000000,0.375097,4320
2,F4Z__liquidation__liq_activity_severity_histor...,-0.105912,-0.342962,0.131825,1.000000,0.356258,4320
3,MEAN168H__F4Z__liquidation__liq_temporal_conce...,0.134841,0.205469,0.073999,1.000000,0.231780,4320
4,HIST__q75_30d,-0.093983,-0.212844,0.054846,1.000000,0.230855,4320
5,STD168H__F4Z__volatility__realized_variance_le...,-0.220717,-0.183353,0.130222,1.000000,0.225510,4320
6,STD24H__F4Z__flow__flow_activity_percentile__5...,-0.175821,-0.193161,0.146644,1.000000,0.222452,4320
7,HIST__mean_30d,-0.072931,-0.209954,0.059240,1.000000,0.222262,4320
8,HIST__median_30d,-0.160471,-0.174639,0.051057,1.000000,0.209651,4320
9,F4Z__liquidation__cross_asset_liq_activity_raw...,-0.050001,-0.208181,0.136586,1.000000,0.207022,4320


## Verification, uncertainty, and fit ledger

In [5]:
print(json.dumps(json.loads((OUTPUT / 'VERIFICATION.json').read_text()), indent=2))
display(pd.DataFrame(json.loads((OUTPUT / 'paired_intervals.json').read_text())))
ledger = pd.DataFrame(json.loads((OUTPUT / 'fit_ledger.json').read_text()))
display(ledger[['id','train_rows','status','seconds']])


{
  "status": "PASS",
  "rows": 3937,
  "max_replay_error": 8.526512829121202e-13,
  "outer_coverage": true,
  "training_windows_and_medians": true,
  "inner_maturity": true,
  "inner_mae_recomputed": true,
  "overall_train_test_mae_recomputed": true,
  "original_manifest_unchanged": true,
  "additional_fits": 0
}


,model,baseline,block_days,low,high
0,nested_standalone_rf,frozen_pair,7,-6.260062,16.336285
1,nested_standalone_rf,previous_rf,7,-4.216233,15.816452
2,nested_standalone_rf,previous_hybrid,7,-5.280218,16.512573
3,nested_standalone_rf,rolling_median,7,-3.180328,15.481841
4,nested_selected,frozen_pair,7,-6.260062,16.336285
5,nested_selected,previous_rf,7,-4.216233,15.816452
6,nested_selected,previous_hybrid,7,-5.280218,16.512573
7,nested_selected,rolling_median,7,-3.180328,15.481841
8,nested_standalone_rf,frozen_pair,14,-9.830442,18.993300
9,nested_standalone_rf,previous_rf,14,-6.583044,18.275402


,id,train_rows,status,seconds
0,2024-08__raw_all,4320,PASS,13.219
1,2024-08__raw_all_deeper,4320,PASS,155.016
2,2024-08__report_only,4320,PASS,34.953
3,2024-08__engineered_ic24,4320,PASS,28.375
4,2024-08__engineered_ic64,4320,PASS,80.094
...,...,...,...,...
59,2024-12__ic64_robust_interactions,4272,PASS,41.454
60,2024-12__residual_ic64,4272,PASS,34.000
61,2025-01__engineered_ic64_log,4320,PASS,3.078
62,2025-01__raw_all,4320,PASS,12.157


## Reproduce without overwriting evidence
Copy the workspace's Code folder, the magnitude mapping JSON in Report/ic_rf_sources, and the complete nested dataset package. Dependencies are recorded in environment.json. For Colab, mount Drive and set WORKSPACE to the copied folder before executing the configuration cell.

The experiment uses the prepared 48-hour dataset; it does not silently switch to another Parquet or target. The original and optimized Colab notebooks and old results remain unchanged.

The following optional cells replay existing saved models or launch a new bounded run. A new run must use a fresh output folder and will spend CPU time. A different output directory does not make previously inspected test history untouched.

In [6]:
import importlib.util
spec = importlib.util.spec_from_file_location('nested_ic_rf_replay', WORKSPACE / 'Code/nested_ic_rf.py')
engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine)
engine.OUT = OUTPUT
VERIFY_SAVED_MODELS = False
if VERIFY_SAVED_MODELS:
    engine.verify()


In [7]:
RUN_NEW_EXPERIMENT = False
NEW_OUTPUT = WORKSPACE / 'Experiments/nested_ic_rf_repeat'
if RUN_NEW_EXPERIMENT:
    assert not NEW_OUTPUT.exists(), 'Choose a new directory; preserve previous evidence.'
    engine.OUT = NEW_OUTPUT
    engine.run()


## Exact training source snapshot
The cell below is a string for inspection, not a training command.

In [8]:
TRAINING_SOURCE = r'''"""IC-guided nested monthly RandomForest experiment; no fits on import."""
from pathlib import Path
import argparse, hashlib, importlib.metadata, itertools, json, sys, time
import numpy as np
import pandas as pd
import joblib
from scipy.stats import spearmanr, pearsonr
from sklearn.ensemble import RandomForestRegressor
from threadpoolctl import threadpool_limits
ROOT=Path(__file__).resolve().parents[1]
PACKAGE=ROOT/'Dataset/TEAM_48H_PLAYGROUND_V1/TEAM_48H_PLAYGROUND_V1'
OUT=ROOT/'Experiments/nested_ic_rf_v3'
sys.path.insert(0,str(PACKAGE/'submissions/member_02'))
import rf_walk_forward as prior
from rf_tuned_walk_forward import histories
from parquet_compat import read_frame
SEED=48
CONFIGS=[]
for name,pool,k,depth,leaf,mf,target,transform,interact in [
 ('raw_all','raw',0,4,64,'sqrt','raw','raw',False),
 ('raw_all_deeper','raw',0,8,32,.3,'raw','raw',False),
 ('report_only','report',0,5,48,.7,'raw','raw',False),
 ('engineered_ic24','engineered',24,4,64,1.,'raw','raw',False),
 ('engineered_ic64','engineered',64,6,32,.7,'raw','raw',False),
 ('engineered_ic128','engineered',128,8,32,.5,'raw','raw',False),
 ('engineered_ic64_log','engineered',64,7,32,.7,'log','raw',False),
 ('engineered_ic64_sqrt','engineered',64,7,48,.7,'sqrt','raw',False),
 ('ic64_rank_interactions','engineered',64,6,48,.7,'raw','rank',True),
 ('report_history','report_history',0,4,64,.7,'raw','raw',False),
 ('ic64_robust_interactions','engineered',64,5,48,.7,'raw','robust',True),
 ('residual_ic64','engineered',64,4,64,.7,'residual','robust',True)]:
 CONFIGS.append(dict(id=name,pool=pool,k=k,depth=depth,leaf=leaf,mf=mf,target=target,transform=transform,interact=interact))

def write(p,obj):
 p.write_text(json.dumps(obj,indent=2,allow_nan=False,default=str),encoding='utf-8')

def load():
 xt,yt,xd,yd=prior.data()
 x=pd.concat([xt,xd]).sort_index().astype('float32')
 y=pd.concat([yt,yd]).sort_index()
 x.index=x.index.as_unit('ns'); y.index=y.index.as_unit('ns')
 y['label_end']=pd.to_datetime(y.label_end,utc=True).astype('datetime64[ns, UTC]')
 assert x.index.equals(y.index) and x.index.is_unique
 assert (y.label_end==y.index+pd.Timedelta(hours=48)).all()
 mapping=json.loads((ROOT/'Report/ic_rf_sources/magnitude_feature_mapping.json').read_text())
 report=[r['column'] for r in mapping if r['mapped']]
 assert len(report)==38 and set(report).issubset(x.columns)
 return x,y,xd.index.as_unit('ns'),report

def engineer(x,y,report):
 x=x.replace([np.inf,-np.inf],np.nan); extra={}
 for c in report:
  s=x[c]; extra['ABS__'+c]=s.abs()
  for h in [1,24]:
   lag=s.shift(freq=pd.Timedelta(hours=h)).reindex(s.index)
   extra[f'LAG{h}H__'+c]=lag
   extra[f'DELTA{h}H__'+c]=s-lag
  for h in [6,24,168]:
   r=s.rolling(f'{h}h',closed='left',min_periods=8); m=r.mean(); sd=r.std()
   extra[f'MEAN{h}H__'+c]=m
   extra[f'STD{h}H__'+c]=sd
   extra[f'INNOV{h}H__'+c]=s-m
   extra[f'Z{h}H__'+c]=((s-m)/sd.clip(lower=1e-4)).clip(-20,20)
 order={'5m':5,'15m':15,'1h':60,'4h':240,'12h':720,'1d':1440}
 for stem in sorted({c.rsplit('__',2)[0] for c in report}):
  cols=[c for c in x if c.rsplit('__',2)[0]==stem and c.split('__')[-2] in order]
  cols.sort(key=lambda c:order[c.split('__')[-2]])
  for a,b in zip(cols[:-1],cols[1:]):
   tag=a+'__VS__'+b.split('__')[-2]
   extra['SLOPE__'+tag]=x[a]-x[b]
   extra['BALANCE__'+tag]=(x[a]-x[b])/(x[a].abs()+x[b].abs()+1.)
 for family in sorted({c.split('__')[1] for c in x}):
  group=x[[c for c in x if c.split('__')[1]==family and 'percentile' in c]]
  if len(group.columns):
   extra[f'FAMILY__{family}__mean_percentile']=group.mean(axis=1)
   extra[f'FAMILY__{family}__dispersion']=group.std(axis=1)
   extra[f'FAMILY__{family}__coverage']=group.notna().mean(axis=1)
 for name,phase,period in [('hour',x.index.hour+x.index.minute/60,24),('weekday',x.index.dayofweek,7)]:
  extra[f'TIME__{name}_sin']=pd.Series(np.sin(2*np.pi*phase/period),index=x.index)
  extra[f'TIME__{name}_cos']=pd.Series(np.cos(2*np.pi*phase/period),index=x.index)
 result=pd.concat([x,pd.DataFrame(extra,index=x.index),histories(y,x.index)],axis=1)
 assert result.columns.is_unique
 return result.replace([np.inf,-np.inf],np.nan).astype('float32')

def ic(a,b):
 a=np.asarray(a); b=np.asarray(b); ok=np.isfinite(a)&np.isfinite(b)
 if ok.sum()<30 or np.unique(a[ok]).size<2 or np.unique(b[ok]).size<2: return 0.
 return float(spearmanr(a[ok],b[ok]).statistic)

def metrics(p,y):
 p=np.asarray(p); y=np.asarray(y)
 return dict(mae=float(np.mean(abs(p-y))),rmse=float(np.sqrt(np.mean((p-y)**2))),
             rank_ic=ic(p,y),pearson_ic=float(pearsonr(p,y).statistic) if np.unique(p).size>1 else None,rows=len(p))

def screen(frame,response):
 admitted=frame.columns[(frame.notna().sum()>=1000)&(frame.nunique()>=2)]
 blocks=frame.index.strftime('%Y-%m'); records=[]; target=response.to_numpy()
 for c in admitted:
  v=frame[c].to_numpy(); full=ic(v,target)
  by=[ic(v[blocks==month],target[blocks==month]) for month in sorted(set(blocks))]
  mean=float(np.mean(by)); sd=float(np.std(by)); agreement=float(np.mean(np.sign(by)==np.sign(mean)))
  records.append(dict(feature=c,rank_ic=full,block_mean_ic=mean,block_std_ic=sd,sign_agreement=agreement,
                      screen_score=abs(mean)*agreement+.25*abs(full)-.1*sd,observations=int(np.isfinite(v).sum())))
 return pd.DataFrame(records).sort_values(['screen_score','feature'],ascending=[False,True]).reset_index(drop=True)

def preprocessing(train,ranking,cfg,report):
 if cfg['pool']=='raw': allowed={c for c in train if c.startswith('F4Z__')}
 elif cfg['pool']=='report': allowed=set(report)
 elif cfg['pool']=='report_history': allowed=set(report)|{c for c in train if c.startswith('HIST__')}
 else: allowed=set(train)
 ranked=[c for c in ranking.feature if c in allowed]
 if cfg['k']:
  pool=ranked[:max(cfg['k']*4,256)]
  matrix=train[pool].rank().corr().abs().fillna(0); cols=[]
  for c in pool:
   if not cols or matrix.loc[c,cols].max()<.97: cols.append(c)
   if len(cols)>=cfg['k']: break
 else: cols=ranked
 if not cols: raise ValueError('No admitted predictors')
 v=train[cols]
 return dict(columns=cols,median=v.median(),lo=v.quantile(.01),hi=v.quantile(.99),
             scale=(v.quantile(.75)-v.quantile(.25)).clip(lower=1e-4),transform=cfg['transform'],
             sorted_values={c:np.sort(v[c].dropna().to_numpy()) for c in cols} if cfg['transform']=='rank' else {},
             interaction_columns=cols[:8] if cfg['interact'] else [])

def design(frame,prep):
 v=frame[prep['columns']]; missing=v.isna().to_numpy(dtype=np.float32); v=v.fillna(prep['median'])
 if prep['transform']=='rank':
  v=pd.DataFrame({c:2*np.searchsorted(prep['sorted_values'][c],v[c],side='right')/len(prep['sorted_values'][c])-1 for c in v},index=v.index)
 elif prep['transform']=='robust':
  v=((v.clip(prep['lo'],prep['hi'],axis=1)-prep['median'])/prep['scale']).clip(-20,20)
  v=np.sign(v)*np.log1p(abs(v))
 arrays=[v.to_numpy(dtype=np.float32),missing]
 if prep['interaction_columns']:
  z=v[prep['interaction_columns']].to_numpy(dtype=np.float32)
  arrays.append(np.column_stack([z[:,i]*z[:,j] for i,j in itertools.combinations(range(z.shape[1]),2)]))
 a=np.column_stack(arrays); assert np.isfinite(a).all()
 return a

def predict(artifact,frame,reference):
 p=artifact['model'].predict(design(frame,artifact['prep'])); target=artifact['config']['target']
 if target=='log': p=np.expm1(p)
 elif target=='sqrt': p=np.square(np.maximum(p,0))
 elif target=='residual': p=p+reference.loc[frame.index].to_numpy()
 return np.maximum(p,0)

def train_keys(y,start):
 return y.index[(y.index>=start-pd.DateOffset(months=3))&(y.label_end<start)]

def month_keys(index,start):
 return index[(index>=start)&(index<start+pd.DateOffset(months=1))]

class Runner:
 def __init__(self,x,y,report,ref):
  self.x=x; self.y=y; self.report=report; self.ref=ref; self.rankings={}; self.receipts=json.loads((OUT/'fit_ledger.json').read_text()) if (OUT/'fit_ledger.json').exists() else []
 def fit_predict(self,cfg,start,keys):
  tag=start.strftime('%Y-%m')+'__'+cfg['id']; path=OUT/'models'/f'{tag}.joblib'
  if path.exists():
   artifact=joblib.load(path)
   return predict(artifact,self.x.loc[keys],self.ref),path
  tk=train_keys(self.y,start)
  assert self.y.loc[tk,'label_end'].max()<start<=keys.min()
  train=self.x.loc[tk]; response=self.y.loc[tk,'magnitude_log_bps']
  sr=response-self.ref.loc[tk] if cfg['target']=='residual' else response
  rk=(str(start),cfg['target']=='residual')
  if rk not in self.rankings:
   self.rankings[rk]=screen(train,sr)
   self.rankings[rk].to_csv(OUT/'ic'/f'{start:%Y-%m}__{cfg["target"] if rk[1] else "magnitude"}.csv',index=False)
  prep=preprocessing(train,self.rankings[rk],cfg,self.report)
  params=dict(n_estimators=96,criterion='squared_error' if cfg['target'] in ['log','sqrt'] else 'absolute_error',
              max_depth=cfg['depth'],min_samples_leaf=cfg['leaf'],max_features=cfg['mf'],
              max_samples=min(2048,len(tk)),bootstrap=True,random_state=SEED,n_jobs=6)
  if len(self.receipts)>=70: raise RuntimeError('Registered fit cap exhausted')
  receipt=dict(id=tag,config=cfg,issuance=str(start),training_start=str(tk.min()),training_end=str(tk.max()),
               max_label_end=str(self.y.loc[tk,'label_end'].max()),train_rows=len(tk),features=prep['columns'],parameters=params,status='STARTED')
  self.receipts.append(receipt); write(OUT/'fit_ledger.json',self.receipts)
  print(f'FIT {len(self.receipts)}/70 {tag} n={len(tk)} p={len(prep["columns"])}',flush=True); tick=time.monotonic()
  try:
   fy=sr if cfg['target']=='residual' else response
   if cfg['target']=='log': fy=np.log1p(fy)
   elif cfg['target']=='sqrt': fy=np.sqrt(fy)
   with threadpool_limits(limits=1):
    model=RandomForestRegressor(**params).fit(design(train,prep),fy)
   artifact=dict(model=model,prep=prep,config=cfg,issuance=start,train_keys=tk)
   result=predict(artifact,self.x.loc[keys],self.ref)
   joblib.dump(artifact,path,compress=3)
   receipt.update(status='PASS',seconds=time.monotonic()-tick)
   write(OUT/'fit_ledger.json',self.receipts)
   print(f'DONE {tag} {receipt["seconds"]:.1f}s',flush=True)
   return result,path
  except BaseException as error:
   receipt.update(status='INTERRUPTED' if isinstance(error,KeyboardInterrupt) else 'FAILED',error=repr(error)); write(OUT/'fit_ledger.json',self.receipts); raise

def calibrate(p,y):
 options=[]
 for scale,offset in itertools.product([.75,.9,1.,1.1,1.25],[-25.,0.,25.]):
  value=np.maximum(0,scale*p+offset)
  options.append((float(np.mean(abs(value-y))),abs(scale-1)+abs(offset)/1000,scale,offset))
 score,_,scale,offset=min(options)
 return dict(scale=scale,offset=offset,mae=score)

def read_csv(path):
 f=pd.read_csv(path,index_col='decision_time'); f.index=pd.to_datetime(f.index,utc=True).as_unit('ns')
 return f

def register(x,months,report,resume=False):
 if OUT.exists() and resume:
  contract=json.loads((OUT/'CONTRACT.json').read_text())
  assert contract['configs']==CONFIGS and contract['outer_months']==list(map(str,months))
  assert (OUT/'RESUME.json').exists(), 'Resume must be explicitly recorded'
  return
 OUT.mkdir(parents=True,exist_ok=False)
 for name in ['models','ic','inner','outer']: (OUT/name).mkdir()
 write(OUT/'CONTRACT.json',dict(created_utc=pd.Timestamp.now(tz='UTC'),outer_months=list(map(str,months)),
  authorization='User requested IC-guided engineering, nested walk-forward and training/testing scores.',
  protocol='Final 3 represented calendar months as outer tests, monthly refits; each outer selects on 3 preceding monthly inner folds.',
  training='Each fit uses preceding 3 calendar months, label_end strictly before issuance.',
  inner_maturity='Inner validation labels also require label_end strictly before outer issuance.',
  features='All 522 existing masked F4Z candidates plus magnitude-report-derived causal engineering and matured history.',
  feature_selection='Pairwise-complete Spearman IC and monthly sign stability; >=1000 finite varied readings per fit; rank-correlation redundancy cutoff .97.',
  configs=CONFIGS,fit_cap=70,caching='Same config/issuance reuses identical fits across overlapping folds.',
  selection='Inner original-scale pooled MAE, tie by rank IC/name. Calibrate scale/offset and compare top-three standalone RF average inside inner layer.',
  calibration_scales=[.75,.9,1.,1.1,1.25],calibration_offsets=[-25.,0.,25.],
  residual_candidate='Explicit frozen-baseline plus RF residual hybrid, separately labeled from standalone RF.',
  overall_training='Pooled in-sample predictions from monthly fitted models; repeated timestamps across overlapping training windows are retained and disclosed.',
  overall_testing='Unique outer timestamp predictions, pooled once; all comparators scored on identical keys.',
  prior_exposure='Historical development and source feature rankings were previously exposed. Nested selection does not create an untouched holdout.',
  stop='One bounded run; no outer-test-driven retuning or guaranteed win.',report_columns=report,engineered_count=len(x.columns),
  original_leaderboard_eligible=False))
 write(OUT/'environment.json',{m:importlib.metadata.version(m) for m in ['numpy','pandas','scipy','scikit-learn','joblib','duckdb']})
 paths=[Path(__file__),PACKAGE/'MANIFEST.json',ROOT/'Report/ic_rf_sources/magnitude_feature_mapping.json']
 write(OUT/'source_hashes.json',{str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths})
 (OUT/'training_source_snapshot.py').write_bytes(Path(__file__).read_bytes())

def assemble_predictions(predictions,s):
 if 'rf_top3_ensemble' in [s['selected_standalone'],s['selected_forecast']]:
  predictions['rf_top3_ensemble']=np.mean([predictions[name] for name in s['top_three']],axis=0)
 result={'nested_raw_rf':predictions[s['best_raw_rf']]}
 for column,name in [('nested_standalone_rf',s['selected_standalone']),('nested_selected',s['selected_forecast'])]:
  c=s['calibrations'][name]; result[column]=np.maximum(0,c['scale']*predictions[name]+c['offset'])
 return result

def run(resume=False):
 raw,y,dev,report=load()
 last=dev.max().normalize().replace(day=1); months=pd.date_range(last-pd.DateOffset(months=2),last,freq='MS')
 print('OUTER MONTHS',list(map(str,months)),flush=True)
 ref=prior.frozen_predictions(raw).magnitude_log_bps
 x=engineer(raw,y,report); register(x,months,report,resume=resume)
 cutoff=months[0]; changed=y.copy(); changed.loc[changed.label_end>=cutoff,'magnitude_log_bps']+=100000
 earlier=raw.index[raw.index<=cutoff]
 np.testing.assert_allclose(histories(y,earlier),histories(changed,earlier),rtol=0,atol=0,equal_nan=True)
 np.testing.assert_allclose(x.loc[earlier],engineer(raw.loc[earlier],y.loc[earlier],report),rtol=0,atol=0,equal_nan=True)
 write(OUT/'PREFLIGHT.json',dict(status='PASS',future_label_perturbation=True,prefix_invariance=True,features=len(x.columns)))
 runner=Runner(x,y,report,ref); results=[]; train_results=[]; score_rows=[]
 for outer in months:
  print('OUTER',str(outer),flush=True)
  starts=pd.date_range(outer-pd.DateOffset(months=3),outer-pd.DateOffset(months=1),freq='MS'); inner=[]
  for start in starts:
   keys=month_keys(x.index,start); keys=keys[y.loc[keys,'label_end']<outer]
   assert y.loc[keys,'label_end'].max()<outer
   block=pd.DataFrame(index=keys); block['truth']=y.loc[keys,'magnitude_log_bps']
   for cfg in CONFIGS: block[cfg['id']],_=runner.fit_predict(cfg,start,keys)
   inner.append(block)
  val=pd.concat(inner).sort_index(); target=val.truth.to_numpy()
  val.to_csv(OUT/'inner'/f'{outer:%Y-%m}_predictions.csv',index_label='decision_time')
  scores={c['id']:metrics(val[c['id']],target) for c in CONFIGS}
  pure=[c['id'] for c in CONFIGS if c['target']!='residual']
  ranked=sorted(pure,key=lambda c:(scores[c]['mae'],-scores[c]['rank_ic'],c)); top=ranked[:3]
  candidates={name:val[name].to_numpy() for name in pure}
  candidates['rf_top3_ensemble']=val[top].mean(axis=1).to_numpy()
  candidates['residual_ic64']=val.residual_ic64.to_numpy()
  cal={name:calibrate(value,target) for name,value in candidates.items()}
  selected_pure=min(pure+['rf_top3_ensemble'],key=lambda c:(cal[c]['mae'],c))
  selected=min(candidates,key=lambda c:(cal[c]['mae'],c))
  s=dict(outer=str(outer),inner_starts=list(map(str,starts)),inner_rows=len(val),
         latest_inner_label_end=str(y.loc[val.index,'label_end'].max()),scores=scores,calibrations=cal,
         top_three=top,best_raw_rf=ranked[0],selected_standalone=selected_pure,
         selected_forecast=selected,selected_is_hybrid=selected=='residual_ic64',frozen_before_outer_scoring=True)
  write(OUT/'outer'/f'{outer:%Y-%m}_selection.json',s)
  print('SELECTED',outer.strftime('%Y-%m'),selected,'standalone',selected_pure,flush=True)
  keys=month_keys(dev,outer); tk=train_keys(y,outer)
  required={ranked[0],selected_pure,selected}
  if 'rf_top3_ensemble' in required: required.update(top); required.remove('rf_top3_ensemble')
  pp={}; tp={}; paths={}
  for name in sorted(required):
   cfg=next(c for c in CONFIGS if c['id']==name)
   pp[name],path=runner.fit_predict(cfg,outer,keys)
   artifact=joblib.load(path); tp[name]=predict(artifact,x.loc[tk],ref); paths[name]=str(path.relative_to(OUT))
  test=pd.DataFrame(assemble_predictions(pp,s),index=keys)
  train=pd.DataFrame(assemble_predictions(tp,s),index=tk)
  inn=pd.DataFrame(assemble_predictions({name:candidates[name] for name in candidates},s),index=val.index)
  for frame,split,true in [(test,'test',y.loc[keys,'magnitude_log_bps']),(train,'train_in_sample',y.loc[tk,'magnitude_log_bps']),(inn,'inner_validation',val.truth)]:
   for name in frame:
    score_rows.append(dict(month=outer.strftime('%Y-%m'),split=split,model=name,**metrics(frame[name],true)))
  train['truth']=y.loc[tk,'magnitude_log_bps']; train['outer_month']=outer.strftime('%Y-%m'); train_results.append(train)
  test['selected_is_hybrid']=selected=='residual_ic64'; test['month']=outer.strftime('%Y-%m')
  test.to_csv(OUT/'outer'/f'{outer:%Y-%m}_predictions.csv',index_label='decision_time')
  s['model_paths']=paths; write(OUT/'outer'/f'{outer:%Y-%m}_selection.json',s); results.append(test)
 predictions=pd.concat(results).sort_index(); expected=dev[dev>=months[0]]
 assert predictions.index.equals(expected) and predictions.index.is_unique
 predictions.to_csv(OUT/'predictions.csv',index_label='decision_time')
 training=pd.concat(train_results); training.to_csv(OUT/'training_predictions.csv',index_label='decision_time')
 pd.DataFrame(score_rows).to_csv(OUT/'train_validation_test_by_month.csv',index=False)
 overall=[]
 for name in ['nested_raw_rf','nested_standalone_rf','nested_selected']:
  overall.append(dict(split='train_in_sample_pooled',model=name,**metrics(training[name],training.truth)))
  overall.append(dict(split='test_outer_unique',model=name,**metrics(predictions[name],y.loc[expected,'magnitude_log_bps'])))
 pd.DataFrame(overall).to_csv(OUT/'overall_training_testing.csv',index=False)
 evaluate(predictions,y,ref); verify()

def evaluate(p,y,ref):
 keys=p.index; truth=y.loc[keys,'magnitude_log_bps']; values=p[['nested_raw_rf','nested_standalone_rf','nested_selected']].copy()
 values['frozen_pair']=ref.loc[keys]
 values['previous_rf']=read_csv(prior.OUT/'predictions.csv').loc[keys,'magnitude_log_bps']
 values['previous_hybrid']=read_csv(PACKAGE/'submissions/member_02/rf_walk_forward_v2/predictions.csv').loc[keys,'magnitude_log_bps']
 values['fixed_M0']=read_frame(PACKAGE/'benchmarks/development_predictions.parquet').loc[keys,'M0_magnitude']
 for month in p.month.unique():
  start=pd.Timestamp(month,tz='UTC'); tk=train_keys(y,start)
  values.loc[p.month==month,'rolling_median']=float(y.loc[tk,'magnitude_log_bps'].median())
 assert values.notna().all().all()
 loss=values.sub(truth,axis=0).abs()
 scoreboard=pd.DataFrame([dict(model=name,**metrics(values[name],truth)) for name in values]).set_index('model').sort_values('mae')
 scoreboard.to_csv(OUT/'scoreboard.csv')
 loss.groupby(p.month).mean().to_csv(OUT/'monthly_mae.csv')
 pd.DataFrame({month:{name:ic(values.loc[p.month==month,name],truth.loc[p.month==month]) for name in values} for month in p.month.unique()}).T.to_csv(OUT/'monthly_rank_ic.csv')
 values.assign(truth=truth).to_csv(OUT/'all_predictions.csv',index_label='decision_time')
 intervals=[]
 for days in [7,14]:
  groups=((keys-keys.min()).total_seconds()//(days*86400)).astype(int)
  sizes=loss.groupby(groups).size().to_numpy(); draws=np.random.default_rng(SEED+days).integers(0,len(sizes),(3000,len(sizes)))
  for model in ['nested_standalone_rf','nested_selected']:
   for baseline in ['frozen_pair','previous_rf','previous_hybrid','rolling_median']:
    sums=(loss[model]-loss[baseline]).groupby(groups).sum().to_numpy()
    lo,hi=np.quantile(sums[draws].sum(axis=1)/sizes[draws].sum(axis=1),[.025,.975])
    intervals.append(dict(model=model,baseline=baseline,block_days=days,low=float(lo),high=float(hi)))
 write(OUT/'paired_intervals.json',intervals)
 summary=dict(status='EXECUTED_ON_REAL_DATA',rows=len(keys),start=str(keys.min()),end=str(keys.max()),
  fit_count=len(json.loads((OUT/'fit_ledger.json').read_text())),scores=[dict(model=name,**metrics(values[name],truth)) for name in scoreboard.index],
  comparisons={model:{base:dict(mae_reduction=float(loss[base].mean()-loss[model].mean()),improvement_percent=float(100*(1-loss[model].mean()/loss[base].mean())),beats=bool(loss[model].mean()<loss[base].mean())) for base in ['frozen_pair','previous_rf','previous_hybrid']} for model in ['nested_standalone_rf','nested_selected']},
  outer_months_with_hybrid=int(p.groupby('month').selected_is_hybrid.first().sum()),
  interpretation='Nested selection; historically exposed dataset and report priors. Nominal intervals, not untouched confirmation. Latest month partial.')
 write(OUT/'RESULT.json',summary)
 print('FINAL SCOREBOARD\n'+scoreboard.to_string(),flush=True)

def verify():
 raw,y,dev,report=load(); x=engineer(raw,y,report); ref=prior.frozen_predictions(raw).magnitude_log_bps
 saved=read_csv(OUT/'predictions.csv'); contract=json.loads((OUT/'CONTRACT.json').read_text())
 assert saved.index.equals(dev[dev>=pd.Timestamp(contract['outer_months'][0])])
 maximum=0.
 for path in sorted((OUT/'outer').glob('*_selection.json')):
  s=json.loads(path.read_text()); start=pd.Timestamp(s['outer']); keys=month_keys(saved.index,start)
  assert pd.Timestamp(s['latest_inner_label_end'])<start
  pp={}
  for name,modelpath in s['model_paths'].items():
   a=joblib.load(OUT/modelpath); expected=train_keys(y,start); assert a['train_keys'].equals(expected)
   np.testing.assert_allclose(a['prep']['median'],x.loc[expected,a['prep']['columns']].median(),rtol=0,atol=0)
   pp[name]=predict(a,x.loc[keys],ref)
  for column,value in assemble_predictions(pp,s).items():
   error=float(np.max(abs(saved.loc[keys,column].to_numpy()-value))); maximum=max(maximum,error); assert error<1e-9
  val=read_csv(OUT/'inner'/f'{start:%Y-%m}_predictions.csv')
  assert (y.loc[val.index,'label_end']<start).all()
  for name,score in s['scores'].items(): assert abs(float(abs(val[name]-val.truth).mean())-score['mae'])<1e-9
 for r in json.loads((OUT/'fit_ledger.json').read_text()):
  assert r['status'] in ['PASS','INTERRUPTED'] and pd.Timestamp(r['max_label_end'])<pd.Timestamp(r['issuance'])
 for path,digest in json.loads((OUT/'source_hashes.json').read_text()).items(): assert hashlib.sha256(Path(path).read_bytes()).hexdigest()==digest
 for path,digest in json.loads((PACKAGE/'MANIFEST.json').read_text()).items(): assert hashlib.sha256((PACKAGE/path).read_bytes()).hexdigest()==digest
 overall=pd.read_csv(OUT/'overall_training_testing.csv'); training=read_csv(OUT/'training_predictions.csv')
 for _,r in overall.iterrows():
  p=training[r.model] if r.split=='train_in_sample_pooled' else saved[r.model]
  t=training.truth if r.split=='train_in_sample_pooled' else y.loc[saved.index,'magnitude_log_bps']
  assert abs(metrics(p,t)['mae']-r.mae)<1e-9
 write(OUT/'VERIFICATION.json',dict(status='PASS',rows=len(saved),max_replay_error=maximum,
  outer_coverage=True,training_windows_and_medians=True,inner_maturity=True,inner_mae_recomputed=True,
  overall_train_test_mae_recomputed=True,original_manifest_unchanged=True,additional_fits=0))
 print('VERIFICATION PASS',flush=True)

if __name__=='__main__':
 parser=argparse.ArgumentParser(); parser.add_argument('command',choices=['run','resume','verify']); args=parser.parse_args()
 verify() if args.command=='verify' else run(resume=args.command=='resume')
'''
print('Loaded exact source text; no training executed.')

Loaded exact source text; no training executed.
